# Lecture 4.7 — Token Cost Tracking and Usage Metadata

**Section 04 — Running Agents, Results & Streaming**

This notebook is the final lecture of Section 4. You will learn how the Agents SDK tracks
token usage for every run, how to read the `Usage` object end to end, how prompt caching
and reasoning tokens show up in that object, and how to build a practical cost tracking
pipeline you can drop into a real project.

## Cell 1 — Install the OpenAI Agents SDK

This notebook uses the `openai-agents` package, the official Python SDK for building
agents with OpenAI models. The cell below installs a pinned version so that the token
counts and field names you see in this notebook match what is documented here.

If the package is already present in this Colab session, the install completes almost
instantly and simply confirms the version is correct.

In [1]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.18.2 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 874.3/874.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 1.8 MB/s eta 0:00:00


## Cell 2 — Configure Your OpenAI API Key

The SDK needs an OpenAI API key to make requests. This notebook uses **Colab Secrets**
exclusively, which keeps the key out of the notebook file itself.

**To add your key in Colab:**

1. Click the key icon (🔑) in the left sidebar to open the **Secrets** panel.
2. Click **Add new secret**.
3. Set the name to `OPENAI_API_KEY` and paste your key as the value.
4. Toggle **Notebook access** on for this notebook.

The cell below reads that secret and writes it into an environment variable, which is
the format the SDK expects.

**Local users:** if you are running this outside Colab, set the environment variable in
your terminal instead, for example `export OPENAI_API_KEY="sk-..."`, and skip the
`userdata` call below.

In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3 — Set the Model Name

Every `Agent` in this notebook references a single `MODEL_NAME` variable rather than a
hardcoded model string. Changing this one line updates the model used everywhere in the
notebook, which makes it easy to swap models later without hunting through every cell.

| Variable | Value | Used for |
|---|---|---|
| `MODEL_NAME` | `"gpt-5.4-mini"` | Default model for most cells in this notebook |
| `REASONING_MODEL` | `"gpt-5.5"` (defined later, Cell 9) | The higher-quality model used specifically to demonstrate reasoning tokens |

In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## Cell 4 — Imports

This cell imports everything the notebook needs. A few of these are easy to get wrong,
so pay attention to where each one comes from:

| Import | Comes from | Why |
|---|---|---|
| `InputTokensDetails`, `OutputTokensDetails` | `openai.types.responses.response_usage` | **Not** from `agents`. These are OpenAI SDK types that the Agents SDK reuses to describe cached and reasoning token counts. |
| `Reasoning` | `openai.types.shared` | Also not from `agents`. Used to set `reasoning.effort` on GPT-5 models. |
| `Agent`, `ModelSettings`, `RunConfig`, `Runner`, `Usage`, `function_tool` | `agents` | The core building blocks you have used throughout Section 4, plus `Usage`, which is new this lecture. |

The most common mistake at this point in the course is assuming `InputTokensDetails` and
`OutputTokensDetails` live in the `agents` package because `Usage` does. They do not.
`Usage` wraps these OpenAI-native types rather than redefining them.

In [4]:
from openai.types.responses.response_usage import (
    InputTokensDetails,
    OutputTokensDetails,
)
from openai.types.shared import Reasoning
from agents import (
    Agent,
    ModelSettings,
    RunConfig,
    Runner,
    Usage,
    function_tool,
)

## Cell 5 — The Usage Object, Field by Field

Before running any code, it helps to see the whole `Usage` object laid out in one place.
Every field below lives on the `Usage` dataclass and is filled in automatically by the
SDK after each run.

| Field | Type | Description |
|---|---|---|
| `requests` | `int` | Total LLM API calls made during the run. |
| `input_tokens` | `int` | Total input tokens sent, summed across all requests. |
| `output_tokens` | `int` | Total output tokens received, summed across all requests. |
| `total_tokens` | `int` | `input_tokens + output_tokens`. |
| `input_tokens_details` | `InputTokensDetails` | Has one field: `.cached_tokens`, the number of input tokens served from OpenAI's prompt cache. |
| `output_tokens_details` | `OutputTokensDetails` | Has one field: `.reasoning_tokens`, the number of internal chain-of-thought tokens GPT-5 models generate. |
| `request_usage_entries` | `list[RequestUsage]` | One entry per individual LLM call, preserving the per-call breakdown that the totals above flatten away. |

**How you access it.** After any `Runner.run()` call, usage lives on the result's
context wrapper, not on the result itself:

```python
usage = result.context_wrapper.usage
```

Writing `result.usage` will not work. The usage data is attached to the run's context
wrapper because the context wrapper is what threads through the whole run, including
every tool call and every handoff.

**How it aggregates.** `Usage` has an `add()` method: `usage_a.add(usage_b)` mutates
`usage_a` in place, summing every numeric field and merging `request_usage_entries`.
You will use this directly in Cell 10 to total usage across multiple `Runner.run()`
calls.

## Cell 6 — Basic Usage Inspection

This is the simplest possible case: one agent, no tools, a single question. The point of
this cell is to see what a `Usage` object looks like when exactly one LLM call was made.

A few decisions worth noticing in the `Agent` definition:

| Parameter | Value | Why |
|---|---|---|
| `reasoning=Reasoning(effort="none")` | Turns off GPT-5's internal chain-of-thought | Keeps `reasoning_tokens` at zero for this baseline example, and keeps the call fast and cheap. |
| `verbosity="low"` | Asks the model for a concise response | Keeps output tokens small so the numbers are easy to read. |

Run the cell and look at the printed values. With a single, tool-free question you
should see `requests` equal to `1` and exactly one entry in `request_usage_entries`.

In [5]:
agent = Agent(
    name="Simple Agent",
    instructions="You are a helpful assistant. Be concise.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

result = await Runner.run(
    agent,
    "What is the capital of France?",
    run_config=RunConfig(workflow_name="Usage demo - simple"),
)

usage = result.context_wrapper.usage

print(f"Requests: {usage.requests}")
print(f"Input tokens: {usage.input_tokens}")
print(f"Output tokens: {usage.output_tokens}")
print(f"Total tokens: {usage.total_tokens}")
print(
    f"Cached input tokens: "
    f"{usage.input_tokens_details.cached_tokens}"
)
print(
    f"Reasoning tokens: "
    f"{usage.output_tokens_details.reasoning_tokens}"
)
print(
    f"Request entries: "
    f"{len(usage.request_usage_entries)}"
)

Requests: 1
Input tokens: 26
Output tokens: 6
Total tokens: 32
Cached input tokens: 0
Reasoning tokens: 0
Request entries: 1


## Cell 7 — Usage With a Tool-Using Agent

A single-turn, no-tool question makes exactly one LLM call. As soon as an agent needs to
call a tool, that changes: the model has to be called once to decide to call the tool,
and called again after the tool result comes back so it can produce a final answer.

This cell defines a `get_weather` tool with `@function_tool`, attaches it to an agent,
and asks a question that requires two lookups. Watch what happens to `requests` and to
`request_usage_entries` once a tool is involved.

The `Agent` reuses the same `reasoning="none"` and `verbosity="low"` settings as Cell 6,
so any difference in token counts you see is from the tool call itself, not from a
change in model behavior.

In [6]:
@function_tool
def get_weather(city: str) -> str:
    """Returns the current weather for a city.

    Args:
        city: The city to get weather for.
    """
    return f"The weather in {city} is sunny and 24 degrees C."


tool_agent = Agent(
    name="Weather Agent",
    instructions=(
        "You are a weather assistant. "
        "Use the get_weather tool to answer questions."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[get_weather],
)

result = await Runner.run(
    tool_agent,
    "What is the weather in Tokyo and Mumbai?",
    run_config=RunConfig(workflow_name="Usage demo - tools"),
)

usage = result.context_wrapper.usage

print(f"Requests: {usage.requests}")
print(f"Total tokens: {usage.total_tokens}")
print("\nPer-request breakdown:")
for i, entry in enumerate(usage.request_usage_entries):
    print(
        f"  Request {i + 1}: "
        f"in={entry.input_tokens} "
        f"out={entry.output_tokens} "
        f"total={entry.total_tokens}"
    )

Requests: 2
Total tokens: 335

Per-request breakdown:
  Request 1: in=88 out=48 total=136
  Request 2: in=180 out=19 total=199


## Cell 8 — Prompt Caching and `cached_tokens`

OpenAI's API caches repeated prompt content automatically. There is nothing to turn on
and no parameter to set. What you can do is observe its effect through
`input_tokens_details.cached_tokens`.

This cell sends the same prompt to the same agent twice in a row. The second call is a
candidate for a cache hit because a meaningful portion of its input, the system
instructions in particular, is identical to the first call.

A note before you run this: caching is not guaranteed on every call. It depends on
factors outside your control, including how OpenAI's infrastructure routes the request.
Treat `cached_tokens` as something to monitor, not something to assume.

In [7]:
prompt = "What is the capital of Japan?"

result1 = await Runner.run(agent, prompt)
usage1 = result1.context_wrapper.usage
print("First call:")
print(f"  Input tokens: {usage1.input_tokens}")
print(
    f"  Cached tokens: "
    f"{usage1.input_tokens_details.cached_tokens}"
)

result2 = await Runner.run(agent, prompt)
usage2 = result2.context_wrapper.usage
print("\nSecond call (same prompt):")
print(f"  Input tokens: {usage2.input_tokens}")
print(
    f"  Cached tokens: "
    f"{usage2.input_tokens_details.cached_tokens}"
)

print(
    "\nNote: cached_tokens may be non-zero on the second "
    "call if eligible for prompt caching. "
    "Cached tokens are charged at a reduced rate."
)

First call:
  Input tokens: 26
  Cached tokens: 0

Second call (same prompt):
  Input tokens: 26
  Cached tokens: 0

Note: cached_tokens may be non-zero on the second call if eligible for prompt caching. Cached tokens are charged at a reduced rate.


## Cell 9 — Reasoning Tokens on GPT-5 Models

GPT-5 models can generate internal chain-of-thought before producing their visible
response. Those internal tokens are tracked separately in
`output_tokens_details.reasoning_tokens`, but they are not free: they are already
included inside `output_tokens`, and you are billed for them.

This cell introduces a second model, `REASONING_MODEL`, set to `"gpt-5.5"`, and gives the
agent `reasoning=Reasoning(effort="low")` instead of `"none"` so you can actually see a
non-zero value. The question asks for a small calculation with shown working, which gives
the model a reason to reason before answering.

Watch the relationship between `output_tokens` and `reasoning_tokens` in the printed
output. The cell computes `visible text tokens` by subtracting one from the other, which
is exactly what you are paying for beyond the reasoning itself.

In [8]:
REASONING_MODEL = "gpt-5.5"

reasoning_agent = Agent(
    name="Reasoning Agent",
    instructions="You are a helpful assistant.",
    model=REASONING_MODEL,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="low"),
        verbosity="low",
    ),
)

result = await Runner.run(
    reasoning_agent,
    "What is 17 multiplied by 23? Show your working.",
)

usage = result.context_wrapper.usage

print(f"Input tokens: {usage.input_tokens}")
print(f"Output tokens: {usage.output_tokens}")
print(
    f"Reasoning tokens: "
    f"{usage.output_tokens_details.reasoning_tokens}"
)

non_reasoning = (
    usage.output_tokens
    - usage.output_tokens_details.reasoning_tokens
)
print(f"Visible text tokens: {non_reasoning}")
print(f"Total tokens: {usage.total_tokens}")

Input tokens: 29
Output tokens: 58
Reasoning tokens: 11
Visible text tokens: 47
Total tokens: 87


## Cell 10 — Aggregating Usage Across Multiple Runs With `Usage.add()`

A real conversation usually spans several `Runner.run()` calls, one per turn. Each call
returns its own `Usage` object scoped to that single run. To get a running total across
a whole conversation, you need to aggregate them yourself, and `Usage.add()` is the tool
built for exactly that.

The pattern below starts from an empty `Usage()`, which has every field at zero, and
calls `.add()` on it after every turn. `add()` mutates the object in place: it sums every
numeric field and merges `request_usage_entries`, so by the end `total_usage` reflects
every LLM call made across all three turns, tool calls included.

This cell also carries the conversation forward between turns using
`result.to_input_list()`, a pattern from Lecture 4.1, so the `tool_agent` from Cell 7 has
the prior turns as context on each new call.

In [9]:
total_usage = Usage()

inputs = [
    "What is the weather in London?",
    "What about Tokyo?",
    "And Sydney?",
]

conversation_history = []
for i, user_input in enumerate(inputs):
    full_input = conversation_history + [
        {"role": "user", "content": user_input}
    ]
    result = await Runner.run(tool_agent, full_input)
    turn_usage = result.context_wrapper.usage
    total_usage.add(turn_usage)
    conversation_history = result.to_input_list()
    print(
        f"Turn {i + 1}: "
        f"{turn_usage.total_tokens} tokens "
        f"({turn_usage.requests} requests)"
    )

print(f"\nTotal across {len(inputs)} turns:")
print(f"  Requests: {total_usage.requests}")
print(f"  Total tokens: {total_usage.total_tokens}")
print(
    f"  Request entries: "
    f"{len(total_usage.request_usage_entries)}"
)

Turn 1: 243 tokens (2 requests)
Turn 2: 369 tokens (2 requests)
Turn 3: 493 tokens (2 requests)

Total across 3 turns:
  Requests: 6
  Total tokens: 1105
  Request entries: 6


## Cell 11 — Building a Cost Calculator

Everything so far has produced token counts. This cell turns those counts into an
estimated dollar figure, which is the pattern you would actually plug into a logging
pipeline or a per-user billing report.

The pricing constants below are illustrative. Always check
[platform.openai.com/pricing](https://platform.openai.com/pricing) for current rates
before using this in a real system, since prices change and vary by model.

The `calculate_cost` function separates cached input tokens from non-cached input
tokens, because they are billed at different rates, then prices output tokens on top.
This is the same three-way split you saw across Cells 6 through 9, now expressed as a
single reusable function.

In [10]:
# Illustrative pricing. Always check platform.openai.com/pricing
# gpt-5.4-mini approximate rates per 1M tokens
PRICE_PER_1M_INPUT = 0.75     # USD
PRICE_PER_1M_OUTPUT = 4.5    # USD
PRICE_PER_1M_CACHED = 0.075    # USD


def calculate_cost(usage: Usage) -> dict:
    """Calculate estimated cost from a Usage object."""
    cached = usage.input_tokens_details.cached_tokens or 0
    non_cached_input = usage.input_tokens - cached

    cost_input = (non_cached_input / 1_000_000) * PRICE_PER_1M_INPUT
    cost_cached = (cached / 1_000_000) * PRICE_PER_1M_CACHED
    cost_output = (usage.output_tokens / 1_000_000) * PRICE_PER_1M_OUTPUT
    total_cost = cost_input + cost_cached + cost_output

    return {
        "input_tokens": usage.input_tokens,
        "cached_tokens": cached,
        "output_tokens": usage.output_tokens,
        "total_tokens": usage.total_tokens,
        "requests": usage.requests,
        "estimated_cost_usd": round(total_cost, 6),
    }


result = await Runner.run(
    tool_agent,
    "What is the weather in Paris?",
)

cost_report = calculate_cost(result.context_wrapper.usage)

print("Cost report:")
for k, v in cost_report.items():
    print(f"  {k}: {v}")

Cost report:
  input_tokens: 213
  cached_tokens: 0
  output_tokens: 30
  total_tokens: 243
  requests: 2
  estimated_cost_usd: 0.000295


## Cell 12 — A Note on Per-Agent Usage

`result.context_wrapper.usage` is a single, aggregated total across the **entire run**,
including every sub-agent invoked through `as_tool()`. The `Usage` object has no
built-in way to split that total back out by agent.

If you need a per-agent cost breakdown, there are two paths worth knowing about, neither
of which this notebook implements:

- **`AgentHooks.on_end()`**, which lets you capture a usage snapshot at the moment each
  agent finishes, so you can compute the delta between snapshots. This is covered in
  Lecture 6.4.
- **`request_usage_entries`**, which you already have. It gives you a per-LLM-call
  breakdown, and if you correlate those entries with trace spans in the OpenAI dashboard
  at [platform.openai.com/traces](https://platform.openai.com/traces), you can attribute
  individual calls back to the agent that made them, even without built-in support.

For now, treat run-level totals as your primary number, and reach for
`request_usage_entries` plus tracing when you need finer detail.

## Cell 13 — Section 4 Recap

That's Usage done, and with it, all of Section 4. Here is everything the section
covered, lecture by lecture:

| Lecture | What was covered |
|---|---|
| 4.1 | `RunResult` in depth: `new_items`, `RunItem` types, `ItemHelpers`, `to_input_list()`, `final_output_as()` |
| 4.2 | Streaming: `Runner.run_streamed()`, `stream_events()`, the three event types |
| 4.3 | Raw text deltas vs. run item events: function call argument streaming, reasoning deltas |
| 4.4 | `RunConfig`: `max_turns` as a direct parameter, model overrides, `workflow_name`, tracing fields |
| 4.5 | `RunContextWrapper` and dependency injection: `result.context_wrapper`, an early look at `Usage` |
| 4.6 | Run-level exceptions: `MaxTurnsExceeded`, guardrail tripwires, `RunErrorDetails` |
| 4.7 | Token cost tracking: `Usage.add()`, prompt caching, reasoning tokens, building a cost calculator |

Section 5 picks up with multi-agent orchestration and guardrails: handoffs, triage
agents, running agents in parallel, and input and output guardrails, including the
guardrail tripwire mechanism in full depth.